# Object detection fine-tuning with LFM2.5-VL-3B and Halo

This notebook fine-tunes [LFM2.5-VL-3B](https://huggingface.co/LiquidAI/LFM2.5-VL-3B) with a LoRA adapter.

It uses [CPPE-5](https://huggingface.co/datasets/rishitdagli/cppe-5), which contains five classes of personal protective equipment. The notebook makes a fixed split with 850 training images and 150 test images. It evaluates the first 30 test images.

The evaluation matches boxes by class at an intersection over union (IoU) threshold of 0.5. The validated adapter reached an F1 of 45.58%.

| Model | Precision | Recall | F1 |
| --- | ---: | ---: | ---: |
| Base model | 0.00% | 0.00% | 0.00% |
| Fine-tuned adapter | 52.76% | 40.12% | 45.58% |


## Start the Halo notebook server

This notebook uses the public Halo image that matches the detected GPU. It needs one supported NVIDIA GPU and a large data volume.

Run these commands on the host. Replace the three paths before you start the container.

~~~bash
git clone --recurse-submodules https://github.com/whitecircle/halo.git
git clone https://github.com/Liquid4All/cookbook.git

export HALO_DIR=/path/to/halo
export COOKBOOK_DIR=/path/to/cookbook
export DATA_DIR=/path/to/large/volume
GPU_NAME="$(nvidia-smi --query-gpu=name --format=csv,noheader | head -n 1)"
case "$GPU_NAME" in
  *H100*|*H200*) HALO_IMAGE=public.ecr.aws/whitecircle/halo:hopper ;;
  *B200*|*B300*|*GB200*|*GB300*) HALO_IMAGE=public.ecr.aws/whitecircle/halo:blackwell ;;
  *) echo "Unsupported GPU: $GPU_NAME"; exit 1 ;;
esac
export HALO_IMAGE

docker pull "$HALO_IMAGE"
docker run --rm -it --gpus '"device=0"' \
  --ipc=host --shm-size=128g \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  -p 8888:8888 \
  -e HF_HOME=/mnt/hf \
  -e HF_DATASETS_CACHE=/mnt/hf/datasets \
  -e TMPDIR=/mnt/tmp \
  -e HALO_DATA_ROOT=/mnt \
  -e PYTHONPATH=/workspace \
  -v "$HALO_DIR":/workspace \
  -v "$COOKBOOK_DIR":/cookbook \
  -v "$DATA_DIR":/mnt \
  -w /workspace \
  "$HALO_IMAGE" \
  jupyter lab --ip=0.0.0.0 --port=8888 --no-browser \
    --allow-root --notebook-dir=/cookbook/finetuning/notebooks
~~~

Open the URL printed by Jupyter. Then open this notebook.


## Check the runtime

Stop here if CUDA is unavailable.


In [ ]:
from pathlib import Path

import torch

DATA_ROOT = Path("/mnt/lfm25-cppe5")
DATASET_PATH = DATA_ROOT / "cppe5-halo"
CONFIG_PATH = DATA_ROOT / "lfm25-vl-3b-cppe5.yaml"
ADAPTER_PATH = DATA_ROOT / "lfm25-vl-3b-cppe5-halo"
BASELINE_PATH = DATA_ROOT / "baseline.json"
TUNED_PATH = DATA_ROOT / "tuned.json"

assert torch.cuda.is_available(), "CUDA is not available"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print(torch.cuda.get_device_name())


## Prepare CPPE-5

The model returns each class and its normalized bounding box. The fixed seed makes the split reproducible.


In [ ]:
import json

from datasets import DatasetDict, Image, load_dataset

DATASET_ID = "rishitdagli/cppe-5"
CLASS_NAMES = ["coverall", "face shield", "gloves", "goggles", "mask"]
SEED = 1337


def detection_prompt():
    labels = ", ".join(CLASS_NAMES)
    return (
        f"Detect all instances of: {labels}. Response must be a JSON array: "
        '[{"label": "...", "bbox": [x1, y1, x2, y2]}, ...]. '
        "Coordinates are normalized to [0,1]."
    )


def training_messages(row):
    width, height = row["width"], row["height"]
    answer = []
    for category, box in zip(
        row["objects"]["category"], row["objects"]["bbox"], strict=True
    ):
        x, y, box_width, box_height = box
        answer.append(
            {
                "label": CLASS_NAMES[category],
                "bbox": [
                    round(max(0.0, x / width), 4),
                    round(max(0.0, y / height), 4),
                    round(min(1.0, (x + box_width) / width), 4),
                    round(min(1.0, (y + box_height) / height), 4),
                ],
            }
        )
    return [
        {"role": "user", "content": detection_prompt()},
        {
            "role": "assistant",
            "content": json.dumps(answer, separators=(",", ":")),
        },
    ]


def prepare_split(dataset):
    messages = [training_messages(row) for row in dataset]
    return (
        dataset.add_column("messages", messages)
        .select_columns(["image", "messages"])
        .cast_column("image", Image(mode="RGB"))
    )


source = load_dataset(DATASET_ID)
split = source["train"].train_test_split(test_size=0.15, seed=SEED)
formatted = DatasetDict(
    {name: prepare_split(dataset) for name, dataset in split.items()}
)

assert len(formatted["train"]) == 850
assert len(formatted["test"]) == 150
if DATASET_PATH.exists():
    raise FileExistsError(
        f"{DATASET_PATH} already exists. Remove it before you rebuild the split."
    )
formatted.save_to_disk(DATASET_PATH)
print({"train": 850, "test": 150, "seed": SEED})


## Define the evaluation

The evaluation uses greedy generation on 30 test images. It matches predicted boxes to reference boxes from the same class.


In [ ]:
import gc
from collections import defaultdict

from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "LiquidAI/LFM2.5-VL-3B"
EVALUATION_IMAGES = 30
IOU_THRESHOLD = 0.5


def empty_boxes():
    return {label: [] for label in CLASS_NAMES}


def parse_boxes(text):
    start = text.find("[")
    end = text.rfind("]")
    if start < 0 or end < start:
        return empty_boxes()
    try:
        items = json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return empty_boxes()
    if not isinstance(items, list):
        return empty_boxes()

    boxes = empty_boxes()
    for item in items:
        if not isinstance(item, dict) or not isinstance(item.get("bbox"), list):
            continue
        box = item["bbox"]
        if len(box) != 4 or not all(
            isinstance(value, (int, float)) for value in box
        ):
            continue
        label = str(item.get("label", "")).strip().casefold().replace("_", " ")
        if label not in boxes:
            continue
        x1, y1, x2, y2 = (float(value) for value in box)
        if x2 <= x1 or y2 <= y1:
            continue
        boxes[label].append(
            [
                max(0.0, min(1.0, x1)),
                max(0.0, min(1.0, y1)),
                max(0.0, min(1.0, x2)),
                max(0.0, min(1.0, y2)),
            ]
        )
    return boxes


def ground_truth(row):
    width, height = row["width"], row["height"]
    boxes = empty_boxes()
    for category, box in zip(
        row["objects"]["category"], row["objects"]["bbox"], strict=True
    ):
        x, y, box_width, box_height = box
        boxes[CLASS_NAMES[category]].append(
            [
                max(0.0, x / width),
                max(0.0, y / height),
                min(1.0, (x + box_width) / width),
                min(1.0, (y + box_height) / height),
            ]
        )
    return boxes


def box_iou(first, second):
    left = max(first[0], second[0])
    top = max(first[1], second[1])
    right = min(first[2], second[2])
    bottom = min(first[3], second[3])
    intersection = max(0.0, right - left) * max(0.0, bottom - top)
    first_area = (first[2] - first[0]) * (first[3] - first[1])
    second_area = (second[2] - second[0]) * (second[3] - second[1])
    union = first_area + second_area - intersection
    return intersection / union if union > 0 else 0.0


def match_boxes(predicted, expected):
    unmatched = set(range(len(expected)))
    true_positives = 0
    for prediction in predicted:
        if not unmatched:
            break
        best = max(unmatched, key=lambda index: box_iou(prediction, expected[index]))
        if box_iou(prediction, expected[best]) >= IOU_THRESHOLD:
            unmatched.remove(best)
            true_positives += 1
    return true_positives, len(predicted) - true_positives, len(unmatched)


def resolve_adapter(path):
    if (path / "adapter_config.json").is_file():
        return path
    checkpoints = list(path.glob("checkpoint-*/adapter_config.json"))
    if not checkpoints:
        raise FileNotFoundError(f"No adapter found under {path}")
    return max(
        checkpoints,
        key=lambda item: int(item.parent.name.removeprefix("checkpoint-")),
    ).parent


def evaluate(output_path, adapter_path=None):
    dataset = load_dataset(DATASET_ID, split="train")
    dataset = dataset.train_test_split(test_size=0.15, seed=SEED)["test"]
    dataset = dataset.select(range(EVALUATION_IMAGES))
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        attn_implementation="sdpa",
        trust_remote_code=True,
    ).cuda()

    adapter = resolve_adapter(adapter_path) if adapter_path else None
    if adapter:
        model = PeftModel.from_pretrained(model, adapter)
    model.eval()

    records = []
    totals = defaultdict(int)
    for image_index, row in enumerate(dataset):
        message = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": detection_prompt()},
                ],
            }
        ]
        text = processor.apply_chat_template(
            message, tokenize=False, add_generation_prompt=True
        )
        inputs = processor(
            text=[text],
            images=[[row["image"].convert("RGB")]],
            return_tensors="pt",
            add_special_tokens=False,
        ).to("cuda")
        with torch.inference_mode():
            generated = model.generate(
                **inputs, max_new_tokens=512, do_sample=False, use_cache=True
            )
        new_tokens = generated[:, inputs["input_ids"].shape[1] :]
        raw_prediction = processor.batch_decode(
            new_tokens, skip_special_tokens=True
        )[0]
        predicted = parse_boxes(raw_prediction)
        expected = ground_truth(row)

        image_totals = defaultdict(int)
        for label in CLASS_NAMES:
            tp, fp, fn = match_boxes(predicted[label], expected[label])
            totals["tp"] += tp
            totals["fp"] += fp
            totals["fn"] += fn
            image_totals["tp"] += tp
            image_totals["fp"] += fp
            image_totals["fn"] += fn
        records.append(
            {
                "image": image_index,
                "expected": expected,
                "predicted": predicted,
                "tp": image_totals["tp"],
                "fp": image_totals["fp"],
                "fn": image_totals["fn"],
                "raw_prediction": raw_prediction,
            }
        )
        print(f"evaluated {image_index + 1}/{len(dataset)}")

    precision = totals["tp"] / max(1, totals["tp"] + totals["fp"])
    recall = totals["tp"] / max(1, totals["tp"] + totals["fn"])
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    result = {
        "model": MODEL_ID,
        "adapter": str(adapter) if adapter else None,
        "images": len(dataset),
        "true_positives": totals["tp"],
        "false_positives": totals["fp"],
        "false_negatives": totals["fn"],
        "precision_at_iou_0_5": precision,
        "recall_at_iou_0_5": recall,
        "f1_at_iou_0_5": f1,
        "predictions": records,
    }
    output_path.write_text(json.dumps(result, indent=2))

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return result


## Measure the base model


In [ ]:
baseline = evaluate(BASELINE_PATH)
print(
    {
        "precision": baseline["precision_at_iou_0_5"],
        "recall": baseline["recall_at_iou_0_5"],
        "f1": baseline["f1_at_iou_0_5"],
    }
)


## Create the Halo configuration

The adapter targets the vision tower, projector, and language model. Halo trains only assistant tokens.


In [ ]:
CONFIG_PATH.write_text(
    r"""
model_name_or_path: LiquidAI/LFM2.5-VL-3B
trust_remote_code: true
attn_implementation: sdpa

dataset:
- /mnt/lfm25-cppe5/cppe5-halo
conversation_field: messages
images_field: image
test_size: null

train_only_on_completions: true
assistant_message_template: "<|im_start|>assistant\n"
max_length: 4096
packing: false
padding_free: false

use_peft: true
lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target_modules:
- q_proj
- k_proj
- v_proj
- out_proj
- in_proj
- w1
- w2
- w3
- linear_1
- linear_2
- fc1
- fc2
lora_task_type: CAUSAL_LM

per_device_train_batch_size: 1
per_device_eval_batch_size: 1
gradient_accumulation_steps: 8
max_steps: 150
gradient_checkpointing: true
gradient_checkpointing_kwargs:
  use_reentrant: false
optim: adamw_torch_fused
learning_rate: 2.0e-05
weight_decay: 0.01
lr_scheduler_type: cosine
warmup_steps: 10
max_grad_norm: 1.0
bf16: true
seed: 42

output_dir: /mnt/lfm25-cppe5/lfm25-vl-3b-cppe5-halo
save_strategy: steps
save_steps: 150
eval_strategy: "no"
save_total_limit: 1
save_only_model: true
logging_steps: 1
logging_first_step: true
report_to: none

dataloader_num_workers: 0
dataset_num_proc: 1
remove_unused_columns: false
""".lstrip()
)
print(CONFIG_PATH)


## Train the adapter

Halo uses one process for this LoRA run. The validated B300 run completed 150 optimizer steps in 482 seconds.


In [ ]:
!halo launch sft {CONFIG_PATH}


## Measure the fine-tuned model


In [ ]:
tuned = evaluate(TUNED_PATH, ADAPTER_PATH)
gain = tuned["f1_at_iou_0_5"] - baseline["f1_at_iou_0_5"]
print(
    {
        "base_precision": baseline["precision_at_iou_0_5"],
        "tuned_precision": tuned["precision_at_iou_0_5"],
        "base_recall": baseline["recall_at_iou_0_5"],
        "tuned_recall": tuned["recall_at_iou_0_5"],
        "base_f1": baseline["f1_at_iou_0_5"],
        "tuned_f1": tuned["f1_at_iou_0_5"],
        "f1_gain_percentage_points": 100 * gain,
    }
)


## Validated result

The run trained 33.3 million parameters, or 1.06% of the model. It used one NVIDIA B300 GPU.

The adapter reached 45.58% F1 on the 30-image evaluation set. Precision was 52.76%, and recall was 40.12%.
